# Lab 05.2 — Spark e MongoDB (database documentale)

## Introduzione
MongoDB memorizza **documenti JSON** (BSON) con struttura flessibile e annidata.
Il **MongoDB Spark Connector** permette di:

* leggere una collezione come DataFrame (lo schema viene **dedotto** da un campione di documenti);
* spingere filtri e *aggregation pipeline* verso MongoDB;
* scrivere DataFrame (anche con campi annidati e array) come documenti.

Il container `mongo` è raggiungibile dal driver Jupyter all'indirizzo `mongodb://mongo:27017`; il connettore viene scaricato da Maven al primo avvio (volume `ivy-cache`).

### Obiettivi
1. Inizializzare una `SparkSession` con il MongoDB Spark Connector (Step 1).
2. Scrivere documenti annidati (struct e array) in una collezione (Step 2).
3. Leggere una collezione e osservare lo schema dedotto (Step 3).
4. Interrogare campi annidati e array con DataFrame e SQL (Step 4).
5. Sfruttare il pushdown di filtri e aggregation pipeline verso MongoDB (Step 5).
6. Trasformare un DataFrame piatto in documenti per cliente (Step 6).
7. Chiudere la sessione (Step 7).

## Step 1: Inizializzare la SparkSession con il connettore MongoDB

Come nel notebook JDBC, `spark.jars.packages` scarica da Maven il connettore (`mongo-spark-connector_2.13`, la versione Scala corrisponde a quella di Spark 4).
Le proprietà `spark.mongodb.read/write.connection.uri` indicano il server MongoDB per lettura e scrittura.
La funzione helper `mongo()` costruisce un lettore per una collezione, evitando di ripetere formato e opzioni.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

# URI del server MongoDB (host "mongo" = nome del servizio Compose)
MONGO = os.environ.get("MONGO_URI", "mongodb://mongo:27017")

# Sessione Spark: il connettore viene scaricato da Maven al primo avvio
spark = (SparkSession.builder
         .appName("Lab05-MongoDB")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.0.2")
         .config("spark.jars.ivy", "/home/jovyan/.ivy2")   # cache persistente dei pacchetti
         .config("spark.mongodb.read.connection.uri", MONGO)
         .config("spark.mongodb.write.connection.uri", MONGO)
         .getOrCreate())

def mongo(collezione, db="shop"):
    """Lettore per una collezione."""
    return spark.read.format("mongodb").option("database", db).option("collection", collezione)

## Step 2: Scrivere documenti annidati

Costruiamo un DataFrame di recensioni con una colonna `struct` (`prodotto`) e una colonna `array` (`tag`), e lo scriviamo con il formato `mongodb`.
Le colonne `struct` diventano sotto-documenti, gli `array` diventano array BSON. `mode("overwrite")` sostituisce la collezione `recensioni` se esiste già.

In [ ]:
# Recensioni con struttura annidata: prodotto = sotto-documento, tag = array di stringhe
recensioni = spark.createDataFrame([
    (1, "Cliente_001", {"titolo": "Il Signore degli Anelli", "categoria": "Libri"}, 5, ["epico", "classico"]),
    (2, "Cliente_002", {"titolo": "Cuffie Bluetooth",        "categoria": "Elettronica"}, 3, ["audio"]),
    (3, "Cliente_001", {"titolo": "Cuffie Bluetooth",        "categoria": "Elettronica"}, 4, ["audio", "economico"]),
    (4, "Cliente_003", {"titolo": "Pallone da calcio",       "categoria": "Sport"}, 5, []),
    (5, "Cliente_004", {"titolo": "Il Signore degli Anelli", "categoria": "Libri"}, 4, ["lungo"]),
], "id INT, cliente STRING, prodotto STRUCT<titolo: STRING, categoria: STRING>, voto INT, tag ARRAY<STRING>")

# Scrittura nella collezione shop.recensioni
(recensioni.write.format("mongodb").mode("overwrite")
           .option("database", "shop").option("collection", "recensioni").save())
print("Scritti", recensioni.count(), "documenti")

### Punti chiave
- `struct` → sotto-documento, `array` → array BSON: la struttura annidata si conserva in scrittura.
- Il formato è `mongodb`; database e collezione si indicano con le opzioni `database` e `collection`.

## Step 3: Leggere una collezione e osservare lo schema dedotto

MongoDB non ha uno schema fisso: il connettore **deduce** lo schema campionando alcuni documenti della collezione.
Rileggiamo `recensioni` e verifichiamo che `prodotto` sia tornato `struct` e `tag` un `array`; il campo `_id` è aggiunto da MongoDB.

In [ ]:
# Lettura della collezione tramite l'helper definito nello Step 1
rec = mongo("recensioni").load()
rec.printSchema()               # schema dedotto dai documenti
rec.show(truncate=False)

## Step 4: Interrogare campi annidati e array

Una volta caricata, la collezione è un normale DataFrame: i campi annidati si raggiungono con la notazione puntata (`prodotto.categoria`), gli array si "aprono" con `explode`, e si può usare anche SQL registrando una vista temporanea.

In [ ]:
# Filtro su un campo annidato e selezione di campi di sotto-documenti
(rec.where(F.col("prodotto.categoria") == "Elettronica")
    .select("cliente", "prodotto.titolo", "voto").show())

# explode: una riga per ogni tag, poi conteggio dei tag più frequenti
(rec.select(F.explode("tag").alias("tag"))
    .groupBy("tag").count().orderBy(F.desc("count")).show())

# Stesso dato interrogato in SQL tramite una vista temporanea
rec.createOrReplaceTempView("recensioni")
spark.sql("""
    SELECT prodotto.titolo AS titolo, ROUND(AVG(voto), 2) AS voto_medio, COUNT(*) AS n
    FROM recensioni GROUP BY prodotto.titolo ORDER BY voto_medio DESC
""").show(truncate=False)

### Punti chiave
- Lo schema è dedotto da un campione di documenti; i campi annidati si leggono con `colonna.campo`.
- `explode` trasforma un array in righe; funziona anche la vista temporanea con Spark SQL.

## Step 5: Pushdown verso MongoDB

* I **filtri** del DataFrame diventano uno stadio `$match` eseguito da MongoDB.
* Con l'opzione `aggregation.pipeline` si può far eseguire a MongoDB un'intera pipeline di aggregazione: a Spark arriva solo il risultato.

In [ ]:
# Il filtro viene inviato a MongoDB come {"$match": {"voto": {"$gte": 4}}}
rec.where("voto >= 4").select("cliente", "voto").show()

# Pipeline di aggregazione eseguita da MongoDB: tag più usati con voto medio
pipeline = """[
  {"$unwind": "$tag"},
  {"$group": {"_id": "$tag", "n": {"$sum": 1}, "voto_medio": {"$avg": "$voto"}}},
  {"$sort": {"n": -1}}
]"""
mongo("recensioni").option("aggregation.pipeline", pipeline).load().show()

### Punti chiave
- I filtri dei DataFrame diventano `$match`: MongoDB restituisce solo i documenti necessari.
- `aggregation.pipeline` delega a MongoDB l'intera aggregazione.

## Step 6: Da DataFrame "piatto" a documenti

Caso tipico: raggruppiamo righe di acquisti per cliente e salviamo **un documento per cliente** con l'elenco dei suoi acquisti, struttura naturale per MongoDB.
`collect_list(struct(...))` costruisce l'array di sotto-documenti; rinominando la colonna `cliente` in `_id` la usiamo come chiave del documento.

In [ ]:
# Acquisti in forma "piatta": una riga per acquisto
acquisti = spark.createDataFrame([
    ("Cliente_001", "2025-01-10", "Libri", 25.0), ("Cliente_001", "2025-02-03", "Sport", 40.0),
    ("Cliente_002", "2025-01-15", "Elettronica", 120.0), ("Cliente_001", "2025-03-01", "Libri", 15.5),
    ("Cliente_003", "2025-02-20", "Casa", 60.0), ("Cliente_002", "2025-03-11", "Libri", 18.0),
], "cliente STRING, data STRING, categoria STRING, importo DOUBLE")

# Un documento per cliente: totali e array annidato con tutti i suoi acquisti
profili = (acquisti.groupBy("cliente")
           .agg(F.count("*").alias("n_acquisti"),
                F.round(F.sum("importo"), 2).alias("totale"),
                F.collect_list(F.struct("data", "categoria", "importo")).alias("acquisti"))
           .withColumnRenamed("cliente", "_id"))       # _id = chiave del documento

# Scrittura nella collezione shop.profili e rilettura per verifica
profili.write.format("mongodb").mode("overwrite").option("database", "shop").option("collection", "profili").save()
mongo("profili").load().show(truncate=False)

### Punti chiave
- `collect_list(struct(...))` costruisce array di sotto-documenti: il modello naturale per MongoDB.
- La colonna `_id` è la chiave primaria del documento.

## Step 7: Chiudere la SparkSession

Termina l'applicazione Spark e libera le risorse del driver.

In [ ]:
# Chiude la sessione e libera le risorse
spark.stop()

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Connettore | `spark.jars.packages`, `spark.mongodb.read/write.connection.uri` | scaricare il MongoDB Spark Connector e indicare il server |
| Scrittura | `write.format("mongodb").option("database", ...).option("collection", ...).save()` | salvare DataFrame (anche annidati) come documenti |
| Lettura e schema dedotto | `spark.read.format("mongodb").load()`, `printSchema()` | leggere una collezione; schema ricavato da un campione |
| Campi annidati e array | `colonna.campo`, `explode`, vista temporanea + SQL | interrogare documenti strutturati |
| Pushdown | filtri → `$match`; `aggregation.pipeline` | far eseguire filtri e aggregazioni a MongoDB |
| Da piatto a documenti | `groupBy`, `collect_list(struct(...))`, colonna `_id` | costruire un documento per entità |
| Chiusura | `spark.stop()` | terminare la sessione |